<a href="https://colab.research.google.com/github/xmegan10/Reddit-Sentiment-Comment-Prediction/blob/main/Reddit%20Comment%20Sentiment%20Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install numpy
%pip install pandas
%pip install matplotlib
%pip install seaborn
%pip install nltk
%pip install re
%pip install emoji
%pip install contractions
%pip install scikit-learn
%pip install collections
%pip install wordcloud
%pip install pyspark -v

ERROR: Could not find a version that satisfies the requirement re (from versions: none)
ERROR: No matching distribution found for re
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 12.8 MB/s eta 0:00:00
ERROR: Could not find a version that satisfies the requirement collections (from versions: none)
ERROR: No matching distribution found for collections
Using pip 24.1.2 from /usr/local/lib/python3.12/dist-packages/pip (python 3.12)


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.probability import FreqDist
import re
import emoji #dealing with emojis
import contractions

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

from collections import Counter
from wordcloud import WordCloud, STOPWORDS, ImageColorGenerator

nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [3]:
import os
print(f"Current Working Directory: {os.getcwd()}")

Current Working Directory: /content


In [4]:
os.listdir()

['.config', 'sample_data']

In [5]:
import torch
torch.cuda.is_available()

True

In [6]:
from pyspark.sql import SparkSession, Row
spark = SparkSession.builder.appName('Load-jston-to-pyspark-dataframe').getOrCreate()
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
apr_comments = spark.read.json("/content/drive/MyDrive/Colab Notebooks/Reddit Comment Pred/r_AskReddit_commentsApril.jsonl")
apr_posts = spark.read.json("/content/drive/MyDrive/Colab Notebooks/Reddit Comment Pred/r_AskReddit_postsApril.jsonl")

In [9]:
apr_comments = apr_comments.select("*").toPandas()
apr_posts = apr_posts.select("*").toPandas()

# EDA

In [10]:
# head of data set
apr_posts.head()

,_meta,all_awardings,allow_live_comments,approved_at_utc,approved_by,archived,author,author_cakeday,author_flair_background_color,author_flair_css_class,...,total_awards_received,treatment_tags,ups,upvote_ratio,url,user_reports,view_count,visited,websocket_url,wls
0,"{'edited_title': None, 'is_edited': None, 'rem...",[],False,None,None,False,534145,None,None,None,...,0,[],1,1.00,https://www.reddit.com/r/AskReddit/comments/1s...,[],None,False,None,6
1,"{'edited_title': None, 'is_edited': None, 'rem...",[],False,None,None,False,Only_Hotel_7221,None,None,None,...,0,[],100,0.88,https://www.reddit.com/r/AskReddit/comments/1s...,[],None,False,None,6
2,"{'edited_title': None, 'is_edited': None, 'rem...",[],False,None,None,False,jts_14,None,None,None,...,0,[],2,1.00,https://www.reddit.com/r/AskReddit/comments/1s...,[],None,False,wss://k8s-lb.wss.redditmedia.com/link/1s94k0r?...,6
3,"{'edited_title': None, 'is_edited': None, 'rem...",[],False,None,None,False,Captain_CrunchYaAss,None,None,None,...,0,[],3,1.00,https://www.reddit.com/r/AskReddit/comments/1s...,[],None,False,wss://k8s-lb.wss.redditmedia.com/link/1s94kun?...,6
4,"{'edited_title': None, 'is_edited': None, 'rem...",[],False,None,None,False,Agile_Wrangler_6731,None,None,None,...,0,[],48,0.94,https://www.reddit.com/r/AskReddit/comments/1s...,[],None,False,wss://k8s-lb.wss.redditmedia.com/link/1s94lab?...,6


In [11]:
apr_posts.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33200 entries, 0 to 33199
Columns: 109 entries, _meta to wls
dtypes: bool(28), float64(1), int64(13), object(67)
memory usage: 21.4+ MB


In [12]:
apr_posts.columns.tolist()

['_meta',
 'all_awardings',
 'allow_live_comments',
 'approved_at_utc',
 'approved_by',
 'archived',
 'author',
 'author_cakeday',
 'author_flair_background_color',
 'author_flair_css_class',
 'author_flair_richtext',
 'author_flair_template_id',
 'author_flair_text',
 'author_flair_text_color',
 'author_flair_type',
 'author_fullname',
 'author_is_blocked',
 'author_patreon_flair',
 'author_premium',
 'awarders',
 'banned_at_utc',
 'banned_by',
 'can_gild',
 'can_mod_post',
 'category',
 'clicked',
 'content_categories',
 'contest_mode',
 'created',
 'created_utc',
 'discussion_type',
 'distinguished',
 'domain',
 'downs',
 'edited',
 'gilded',
 'hidden',
 'hide_score',
 'id',
 'is_created_from_ads_ui',
 'is_crosspostable',
 'is_meta',
 'is_original_content',
 'is_reddit_media_domain',
 'is_robot_indexable',
 'is_self',
 'is_video',
 'likes',
 'link_flair_background_color',
 'link_flair_css_class',
 'link_flair_richtext',
 'link_flair_template_id',
 'link_flair_text',
 'link_flair_tex

In [13]:
apr_comments.head()

,_meta,all_awardings,approved_at_utc,approved_by,archived,associated_award,author,author_cakeday,author_flair_background_color,author_flair_css_class,...,subreddit,subreddit_id,subreddit_name_prefixed,subreddit_type,top_awarded_type,total_awards_received,treatment_tags,unrepliable_reason,ups,user_reports
0,"{'is_edited': None, 'removal_type': None, 'ret...",[],None,None,False,None,QuayleDan128,None,None,None,...,AskReddit,t5_2qh1i,r/AskReddit,public,None,0,[],None,1,[]
1,"{'is_edited': None, 'removal_type': None, 'ret...",[],None,None,False,None,_persian_princess_,None,None,None,...,AskReddit,t5_2qh1i,r/AskReddit,public,None,0,[],None,3,[]
2,"{'is_edited': None, 'removal_type': None, 'ret...",[],None,None,False,None,anormalgeek,None,None,None,...,AskReddit,t5_2qh1i,r/AskReddit,public,None,0,[],None,4,[]
3,"{'is_edited': None, 'removal_type': None, 'ret...",[],None,None,False,None,skatefan420,None,None,None,...,AskReddit,t5_2qh1i,r/AskReddit,public,None,0,[],None,2,[]
4,"{'is_edited': None, 'removal_type': None, 'ret...",[],None,None,False,None,ThinkIndependently5,None,None,None,...,AskReddit,t5_2qh1i,r/AskReddit,public,None,0,[],None,1,[]


In [14]:
apr_comments.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 833548 entries, 0 to 833547
Data columns (total 73 columns):
 #   Column                           Non-Null Count   Dtype 
---  ------                           --------------   ----- 
 0   _meta                            810063 non-null  object
 1   all_awardings                    833548 non-null  object
 2   approved_at_utc                  0 non-null       object
 3   approved_by                      0 non-null       object
 4   archived                         833548 non-null  bool  
 5   associated_award                 0 non-null       object
 6   author                           833548 non-null  object
 7   author_cakeday                   2567 non-null    object
 8   author_flair_background_color    23900 non-null   object
 9   author_flair_css_class           0 non-null       object
 10  author_flair_richtext            809899 non-null  object
 11  author_flair_template_id         0 non-null       object
 12  author_flair_tex

In [15]:
def create_edadf(*dataframes):
    eda_df = []
    for df in dataframes:
        eda_df.append({
            "name": df.name,
            "num_rows": df.shape[0],
            "num_cols": df.shape[1],
            "contains_null": df.isnull().any(axis = None)
        })
    return pd.DataFrame(eda_df)

p_eda_df = create_edadf(apr_posts)
c_eda_df = create_edadf(apr_comments)

In [17]:
p_eda_df

,name,num_rows,num_cols,contains_null
0,0 t3_1s94jjw 1 t3_1s94jno 2 ...,33200,109,True


In [18]:
c_eda_df

,name,num_rows,num_cols,contains_null
0,0 t1_odlodpk 1 t1_odlodp9 2 ...,833548,73,True


In [16]:
posts_df = apr_posts.copy()
comments_df = apr_comments.copy()

# Split Data into Train and Test

In [17]:
#remove t3 from link_id in comments_df
comments_df['link_id'] = comments_df['link_id'].str.replace('t3_', '')
comments_df['link_id'].value_counts()

,count
link_id,
1sa3jwv,13149
1sdmusm,9136
1scriio,8994
1sbcji8,8162
1sagz0b,7931
...,...
1sass35,1
1sasscy,1
1sase9e,1


In [18]:
from sklearn.model_selection import train_test_split
ptrain_set, ptest_set = train_test_split(posts_df, test_size = 0.2, random_state = 42)

#get the list of IDs for each split
ptrain_ids = ptrain_set['id'].unique()
ptest_ids = ptest_set['id'].unique()

ctrain_set = comments_df[comments_df['link_id'].isin(ptrain_ids)]
ctest_set = comments_df[comments_df['link_id'].isin(ptest_ids)]

In [19]:
ptrain_set.info()

<class 'pandas.core.frame.DataFrame'>
Index: 26560 entries, 18578 to 15795
Columns: 109 entries, _meta to wls
dtypes: bool(28), float64(1), int64(13), object(67)
memory usage: 17.3+ MB


In [23]:
ctrain_set.info()

<class 'pandas.core.frame.DataFrame'>
Index: 644213 entries, 96 to 833545
Data columns (total 73 columns):
 #   Column                           Non-Null Count   Dtype 
---  ------                           --------------   ----- 
 0   _meta                            628259 non-null  object
 1   all_awardings                    644213 non-null  object
 2   approved_at_utc                  0 non-null       object
 3   approved_by                      0 non-null       object
 4   archived                         644213 non-null  bool  
 5   associated_award                 0 non-null       object
 6   author                           644213 non-null  object
 7   author_cakeday                   1993 non-null    object
 8   author_flair_background_color    18005 non-null   object
 9   author_flair_css_class           0 non-null       object
 10  author_flair_richtext            626401 non-null  object
 11  author_flair_template_id         0 non-null       object
 12  author_flair_text   

In [20]:
missing_ids = ctrain_set[~ctrain_set['link_id'].isin(ptrain_set['id'])]['id'].unique()

print(f"Number of unique Post IDs missing from ptrain_setk: {len(missing_ids)}")
print("First 10 missing IDs:", missing_ids[:10])

Number of unique Post IDs missing from ptrain_setk: 0
First 10 missing IDs: []


# Remove Null and One-Unique-only columns


In [21]:
len(ctrain_set.index)

644213

In [22]:
def drop_cols(df):
  for c in df.columns:
    # Check if all values are null
    if df[c].isnull().sum() == len(df.index):
      df.drop(c, axis=1, inplace=True)
    else:
      # Try to check for unique values. If TypeError occurs,
      # convert elements to string representation
      try:
        if len(df[c].unique()) == 1:
          df.drop(c, axis=1, inplace=True)
      except TypeError:
        # Handle unhashable types
        # Convert to string to check if all string representations are the same
        if df[c].astype(str).nunique() == 1:
          df.drop(c, axis=1, inplace=True)
  return df

In [23]:
ptrain_set = drop_cols(ptrain_set)
ctrain_set = drop_cols(ctrain_set)

/tmp/ipykernel_692/2402450421.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop(c, axis=1, inplace=True)
/tmp/ipykernel_692/2402450421.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop(c, axis=1, inplace=True)
/tmp/ipykernel_692/2402450421.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop(c, axis=1, inplace=True)
/tmp/ipykernel_692/2402450421.py:11: SettingWithCopyWarning: 
A value is trying to be set on 

In [24]:
ptrain_set.info()

<class 'pandas.core.frame.DataFrame'>
Index: 26560 entries, 18578 to 15795
Data columns (total 45 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   _meta                          25637 non-null  object 
 1   author                         26560 non-null  object 
 2   author_cakeday                 71 non-null     object 
 3   author_flair_background_color  1572 non-null   object 
 4   author_flair_richtext          24988 non-null  object 
 5   author_flair_text_color        1572 non-null   object 
 6   author_flair_type              24988 non-null  object 
 7   author_fullname                24988 non-null  object 
 8   author_patreon_flair           24988 non-null  object 
 9   author_premium                 24988 non-null  object 
 10  created                        26560 non-null  int64  
 11  created_utc                    26560 non-null  int64  
 12  domain                         26560 non-null  

In [29]:
ctrain_set.info()

<class 'pandas.core.frame.DataFrame'>
Index: 644213 entries, 96 to 833545
Data columns (total 37 columns):
 #   Column                         Non-Null Count   Dtype 
---  ------                         --------------   ----- 
 0   _meta                          628259 non-null  object
 1   author                         644213 non-null  object
 2   author_cakeday                 1993 non-null    object
 3   author_flair_background_color  18005 non-null   object
 4   author_flair_richtext          626401 non-null  object
 5   author_flair_text_color        18005 non-null   object
 6   author_flair_type              626401 non-null  object
 7   author_fullname                626208 non-null  object
 8   author_patreon_flair           626401 non-null  object
 9   author_premium                 626401 non-null  object
 10  body                           644213 non-null  object
 11  collapsed                      644213 non-null  bool  
 12  collapsed_reason               213 non-null     

# Select only relevant columns

In [25]:
ptrain_keep_cols = ["created_utc",
                    "id",
                    "is_crosspostable",
                    "link_flair_text",
                    "link_flair_type",
                    "over_18",
                    "poll_data",
                    "selftext",
                    "spoiler",
                    "thumbnail",
                    "title"]

ctrain_keep_cols = ["body",
                    "created_utc",
                    "distinguished",
                    "id",
                    "link_id",
                    "locked",
                    "parent_id",
                    "score",
                    "stickied",
                    "ups"]

In [26]:
ptrain_set = ptrain_set[ptrain_keep_cols]
ctrain_set = ctrain_set[ctrain_keep_cols]

In [27]:
ptrain_set.info()

<class 'pandas.core.frame.DataFrame'>
Index: 26560 entries, 18578 to 15795
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   created_utc       26560 non-null  int64 
 1   id                26560 non-null  object
 2   is_crosspostable  26560 non-null  bool  
 3   link_flair_text   95 non-null     object
 4   link_flair_type   26560 non-null  object
 5   over_18           26560 non-null  bool  
 6   poll_data         1253 non-null   object
 7   selftext          26560 non-null  object
 8   spoiler           26560 non-null  bool  
 9   thumbnail         26454 non-null  object
 10  title             26560 non-null  object
dtypes: bool(3), int64(1), object(7)
memory usage: 1.9+ MB


In [33]:
ctrain_set.info()

<class 'pandas.core.frame.DataFrame'>
Index: 644213 entries, 96 to 833545
Data columns (total 10 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   body           644213 non-null  object
 1   created_utc    644213 non-null  int64 
 2   distinguished  3917 non-null    object
 3   id             644213 non-null  object
 4   link_id        644213 non-null  object
 5   locked         644213 non-null  bool  
 6   parent_id      644213 non-null  object
 7   score          644213 non-null  int64 
 8   stickied       644213 non-null  bool  
 9   ups            644213 non-null  int64 
dtypes: bool(2), int64(3), object(5)
memory usage: 45.5+ MB


# Preprocess Text

In [28]:
ptrain_set[ptrain_set["selftext"] != ""]["selftext"].value_counts()

,count
selftext,
[removed],1115
[deleted],435
[ Removed by Reddit on account of violating the [content policy](/help/contentpolicy). ],7


In [31]:
#group removed and rename all types
ptrain_set["selftext"] = ptrain_set["selftext"].replace({"":'none','[removed]':'removed','[ Removed by Reddit on account of violating the [content policy](/help/contentpolicy). ]':'removed','[deleted]':'deleted'})

In [33]:
#rename selftext column to status
ptrain_set = ptrain_set.rename(columns = {"selftext":"status"})

In [34]:
ptrain_set["status"].value_counts()

,count
status,
none,25003
removed,1122
deleted,435


In [ ]:
#r/AskReddit does not frequently has posts with bodies, so we skip this code for now. In the future if using this code on a subreddit with long bodies, you can use this code:
#combines selftext (body) with title (post title)

#ptrain_set["post_string"] = ptrain_set["post_title"].str.cat(ptrain_set["post_text"], sep = " ",na_rep = "")
#ptrain_set["post_string"].eq("").sum()

In [ ]:
#add meta data to improve training
#add word counts of post_string as pstring_len
ptrain_set["post_length"] = ptrain_set["post_string"].str.split().str.len()
ptrain_set.head()



In [ ]:
def preprocess_df(df, text_col = "post_string"):
    #CLEAN TEXT
    processed_texts = []

    #loading stop words and objects from classes
    default_stopwords = set(stopwords.words('english'))
    default_stopwords.add("edit")
    lmtzr = WordNetLemmatizer()

    def clean_text(sentence):
        #return "" if sentence is NaN so no error occurs
        if not isinstance(sentence, str) or sentence.strip() == "":
            return ""

        #convert emojis to text
        sentence = emoji.demojize(sentence)
        #remove https from strings
        sentence = re.sub(r'http\S+','', sentence)
        #remove new lines
        sentence = re.sub(r'\\n|:|_',' ', sentence)
        #expand contractions
        sentence = contractions.fix(sentence)
        #remove punctuations and lowercase
        sentence = re.sub(r'[^a-zA-Z0-9\s]', '', sentence).lower()


        #tokenize words
        tokens = word_tokenize(sentence)

        final_words = []
        for word in tokens: #for each word in the sentence
            fixed_word = contractions.fix(word) #expand contractions
            if fixed_word not in default_stopwords:
                lemma = lmtzr.lemmatize(fixed_word) #lemmatize words
                final_words.append(lemma) #append the singular lemmatized word to the final_words list

        return ' '.join(final_words) #append the list of words to processed_text

    df[text_col] = df[text_col].apply(clean_text)
    return df

In [ ]:
ptrain_set = preprocess_df(ptrain_set, text_col = "post_string")

In [ ]:
ptrain_set["post_string"]

In [ ]:
ptrain_set_test = ptrain_set.copy()

## Pipeline: FeaturePreprocessor

In [ ]:
#write pipeline for preprocess step
from sklearn.base import BaseEstimator, TransformerMixin

class FeaturePreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, variables = None):
        self.variables = variables #placeholder for now; change if need to add specific conditions

        self.stop_words = set(stopwords.words('english'))
        self.stop_words.add("edit")
        self.lmtzr = WordNetLemmatizer()

    def fit(self, X, y = None):
        return self

    def clean_text(self, text):
        if not isinstance(text, str) or text.strip() == "":
            return ""

        #convert emojis to text
        text = emoji.demojize(text)
        #expand contractions
        text = contractions.fix(text)
        #remove https from strings
        text = re.sub(r'http\S+','', text)
        #remove new lines
        text = re.sub(r'\\n|:|_',' ', text)
        #remove punctuations and lowercase
        text = re.sub(r'[^a-zA-Z0-9\s]', '', text).lower()

        #tokenize words
        tokens = word_tokenize(text)
        cleaned_tokens = [self.lmtzr.lemmatize(word) for word in tokens if word not in self.stop_words]

        return cleaned_tokens


    def transform(self, X):
        X = X.copy()

        #combine post_title and post_text into post_string
        X["post_string"] = X["post_title"].str.cat(X["post_text"], sep = " ",na_rep = "")
        #add word counts of post_string as pstring_len
        X["pstring_len"] = X["post_string"].str.split().str.len()

        X["post_string"] = X["post_string"].apply(self.clean_text).str.join(" ")

        return X

In [ ]:
feat_preprocessor = FeaturePreprocessor()
ptrain_set_test = feat_preprocessor.transform(ptrain_set_test)